# Supplementary Tables: Ensembl × CAT Gene Concordance

Generates three supplementary TSV tables for publication plus pre-release QC outputs.

| Table | File | Content |
|-------|------|---------|
| S1 | supp_table_s1_gene_concordance_summary.tsv | Per-gene summary across all 462 assemblies |
| S2 | supp_table_s2_full_gene_pairs.tsv | Per-assembly × gene-pair detail |
| S3 | supp_table_s3_cds_reference_discordant_genes.tsv | Extensive CDS/reference-discordant gene source table for the supplementary figure inset |
| QC | pre_release_qc/*.tsv, pre_release_qc/*.png | Release checks and simple diagnostic plots |

**Input:** Pipeline output directory (set OUTPUT_DIR below)

In [ ]:
!pip install -U pandas pyarrow

In [ ]:
import os
import time
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = Path(os.getenv('HPRC_QC_OUTPUT_DIR',
    '/hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results'))

QC_DIR = OUTPUT_DIR / 'qc_metrics'
RESULTS_DIR = OUTPUT_DIR / 'results'
SUPP_DIR = OUTPUT_DIR / 'supplementary_tables'
SUPP_DIR.mkdir(parents=True, exist_ok=True)

# Parquet caches — dramatically faster than re-reading 462 TSV files each run.
# Set HPRC_QC_FORCE_RELOAD=1 to regenerate from raw files after a pipeline re-run.
CACHE_DIR = SUPP_DIR / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FORCE_RELOAD = os.getenv('HPRC_QC_FORCE_RELOAD', '0').strip().lower() in {'1', 'true', 'yes'}

BUILD_S2_FULL = os.getenv('HPRC_QC_BUILD_S2_FULL', '0').strip().lower() in {'1', 'true', 'yes'}

JOIN_COLS = ['assembly_accession', 'ensembl_gene_id', 'cat_gene_id']
TRANSCRIPT_CONCORDANCE_COLS = JOIN_COLS + [
    'n_ensembl_transcripts', 'n_cat_transcripts',
    'n_ens_exact', 'n_cat_exact',
    'ens_to_cat_concordance_rate', 'cat_to_ens_concordance_rate',
    'avg_jaccard_index',
]
CODING_INTEGRITY_COLS = JOIN_COLS + [
    'classification', 'start_codon_match', 'stop_codon_match', 'frameshift_detected',
]
DIVERGENCE_COLS = JOIN_COLS + [
    'gene_name', 'ensembl_biotype', 'is_coding',
    'ens_cds_match', 'cat_cds_match',
    'ens_cds_change_type', 'cat_cds_change_type',
    'divergence_category',
]
RBH_COLS = [
    'ensembl_id', 'cat_id', 'ensembl_name', 'cat_name',
    'ensembl_biotype', 'cat_biotype', 'frac_ensembl_covered',
    'frac_cat_covered', 'is_rbh',
]

print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"QC_DIR exists: {QC_DIR.exists()}")
print(f"RESULTS_DIR exists: {RESULTS_DIR.exists()}")
print(f"CACHE_DIR: {CACHE_DIR}")
print(f"FORCE_RELOAD: {FORCE_RELOAD}")

## Load per-assembly data

In [ ]:
_cache = CACHE_DIR / 'transcript_concordance.parquet'
if _cache.exists() and not FORCE_RELOAD:
    t0 = time.time()
    transcript_concordance_df = pd.read_parquet(_cache, columns=TRANSCRIPT_CONCORDANCE_COLS)
    print(f"Loaded transcript concordance from cache: {len(transcript_concordance_df):,} rows ({time.time()-t0:.1f}s)")
else:
    transcript_concordance_frames = []
    for i, accession_dir in enumerate(sorted(QC_DIR.iterdir())):
        if not accession_dir.is_dir():
            continue
        accession = accession_dir.name
        fp = accession_dir / f'{accession}_transcript_concordance.tsv'
        try:
            df = pd.read_csv(fp, sep='\t', usecols=lambda c: c in TRANSCRIPT_CONCORDANCE_COLS)
            if df.empty:
                print(f"WARNING: empty file {fp}")
                continue
            transcript_concordance_frames.append(df)
        except FileNotFoundError:
            print(f"WARNING: missing file {fp}")
        except Exception as e:
            print(f"WARNING: could not read {fp}: {e}")
        if (i + 1) % 50 == 0:
            print(f"  transcript concordance: {i+1} assemblies loaded...")

    transcript_concordance_df = pd.concat(transcript_concordance_frames, ignore_index=True)
    transcript_concordance_df.to_parquet(_cache, index=False)
    print(f"Loaded transcript concordance: {len(transcript_concordance_df):,} rows from {len(transcript_concordance_frames)} assemblies → cached")

In [ ]:
_cache = CACHE_DIR / 'coding_integrity.parquet'
if _cache.exists() and not FORCE_RELOAD:
    t0 = time.time()
    coding_integrity_df = pd.read_parquet(_cache, columns=CODING_INTEGRITY_COLS)
    print(f"Loaded coding integrity from cache: {len(coding_integrity_df):,} rows ({time.time()-t0:.1f}s)")
else:
    coding_integrity_frames = []
    for i, accession_dir in enumerate(sorted(QC_DIR.iterdir())):
        if not accession_dir.is_dir():
            continue
        accession = accession_dir.name
        fp = accession_dir / f'{accession}_coding_integrity.tsv'
        try:
            df = pd.read_csv(fp, sep='\t', usecols=lambda c: c in CODING_INTEGRITY_COLS)
            if df.empty:
                print(f"WARNING: empty file {fp}")
                continue
            coding_integrity_frames.append(df)
        except FileNotFoundError:
            print(f"WARNING: missing file {fp}")
        except Exception as e:
            print(f"WARNING: could not read {fp}: {e}")
        if (i + 1) % 50 == 0:
            print(f"  coding integrity: {i+1} assemblies loaded...")

    coding_integrity_df = pd.concat(coding_integrity_frames, ignore_index=True)
    coding_integrity_df.to_parquet(_cache, index=False)
    print(f"Loaded coding integrity: {len(coding_integrity_df):,} rows from {len(coding_integrity_frames)} assemblies → cached")

In [ ]:
import gc

def safe_mode(series):
    m = series.dropna().mode()
    return m.iloc[0] if len(m) > 0 else np.nan

# Gene presence files are skipped entirely — all IDs in the pipeline output are
# assembly-specific, so the only canonical gene key is ensembl_name (HGNC symbol)
# from the rbh files. gp_agg, tc_medians, and rbh_name_lookup are all built in
# cell-8 from rbh_df and merged, while those DataFrames are already in memory.
print("Helpers defined.")

In [ ]:
_cache = CACHE_DIR / 'divergence.parquet'
if _cache.exists() and not FORCE_RELOAD:
    t0 = time.time()
    divergence_df = pd.read_parquet(_cache, columns=DIVERGENCE_COLS)
    print(f"Loaded grch38 divergence from cache: {len(divergence_df):,} rows ({time.time()-t0:.1f}s)")
else:
    divergence_frames = []
    for i, accession_dir in enumerate(sorted(QC_DIR.iterdir())):
        if not accession_dir.is_dir():
            continue
        accession = accession_dir.name
        fp = accession_dir / f'{accession}_grch38_divergence.tsv'
        if not fp.exists():
            continue
        try:
            df = pd.read_csv(fp, sep='\t', usecols=lambda c: c in DIVERGENCE_COLS)
            if df.empty:
                print(f"WARNING: empty file {fp}")
                continue
            divergence_frames.append(df)
        except Exception as e:
            print(f"WARNING: could not read {fp}: {e}")
        if (i + 1) % 50 == 0:
            print(f"  divergence: {len(divergence_frames)} assemblies loaded...")

    if divergence_frames:
        divergence_df = pd.concat(divergence_frames, ignore_index=True)
        divergence_df.to_parquet(_cache, index=False)
        print(f"Loaded grch38 divergence: {len(divergence_df):,} rows from {len(divergence_frames)} assemblies → cached")
    else:
        divergence_df = pd.DataFrame(columns=['assembly_accession', 'sample_name', 'ensembl_gene_id',
                                               'cat_gene_id', 'gene_name', 'ensembl_biotype',
                                               'ref_biotype', 'divergence_category'])
        print("WARNING: no grch38 divergence files found; divergence_df is empty")

## Build Supplementary Table S2: Per-assembly gene-pair detail

In [ ]:
t0 = time.time()

# Load all RBH gene pair files.
# rbh files use ensembl_id/cat_id and have no assembly_accession —
# inject it from the directory name and rename to pipeline-wide conventions.
rbh_frames = []
for i, accession_dir in enumerate(sorted(RESULTS_DIR.iterdir())):
    if not accession_dir.is_dir():
        continue
    accession = accession_dir.name
    fp = accession_dir / f'{accession}.gene_pairs_rbh.tsv'
    try:
        df = pd.read_csv(fp, sep='\t', usecols=lambda c: c in RBH_COLS)
        if df.empty:
            print(f"WARNING: empty file {fp}")
            continue
        df['assembly_accession'] = accession
        df = df.rename(columns={'ensembl_id': 'ensembl_gene_id', 'cat_id': 'cat_gene_id'})
        rbh_frames.append(df)
    except FileNotFoundError:
        print(f"WARNING: missing file {fp}")
    except Exception as e:
        print(f"WARNING: could not read {fp}: {e}")
    if (i + 1) % 50 == 0:
        print(f"  RBH pairs: {len(rbh_frames)} assemblies loaded...")

rbh_df = pd.concat(rbh_frames, ignore_index=True)
del rbh_frames
gc.collect()
print(f"Loaded RBH gene pairs: {len(rbh_df):,} rows from {rbh_df['assembly_accession'].nunique()} assemblies ({time.time()-t0:.1f}s)")

# Keep only RBH pairs (all rows should already have is_rbh=True, but filter to be safe)
rbh_df = rbh_df[rbh_df['is_rbh'] == True].copy()
print(f"After filtering is_rbh=True: {len(rbh_df):,} rows")

# --- Build S2_full: per-assembly gene-pair detail (data deposit, not journal supplement) ---

join_cols = JOIN_COLS

# Join transcript concordance
tc_cols = join_cols + [
    'n_ensembl_transcripts', 'n_cat_transcripts',
    'n_ens_exact', 'n_cat_exact',
    'ens_to_cat_concordance_rate', 'cat_to_ens_concordance_rate',
    'avg_jaccard_index',
]
tc_cols = [c for c in tc_cols if c in transcript_concordance_df.columns]
tc_subset = transcript_concordance_df[tc_cols].drop_duplicates(subset=join_cols)
merged = rbh_df.merge(tc_subset, on=join_cols, how='left', copy=False)
del tc_subset
gc.collect()
print(f"After joining transcript concordance: {len(merged):,} rows")

# Join coding integrity
ci_cols = join_cols + ['classification', 'start_codon_match', 'stop_codon_match', 'frameshift_detected']
ci_cols = [c for c in ci_cols if c in coding_integrity_df.columns]
ci_subset = coding_integrity_df[ci_cols].drop_duplicates(subset=join_cols)
ci_subset = ci_subset.rename(columns={'classification': 'cds_classification'})
merged = merged.merge(ci_subset, on=join_cols, how='left', copy=False)
del ci_subset
gc.collect()
print(f"After joining coding integrity: {len(merged):,} rows")

# Join divergence
if not divergence_df.empty:
    div_cols = [c for c in join_cols + ['divergence_category'] if c in divergence_df.columns]
    div_subset = divergence_df[div_cols].drop_duplicates(subset=join_cols)
    merged = merged.merge(div_subset, on=join_cols, how='left', copy=False)
    del div_subset
    gc.collect()
else:
    merged['divergence_category'] = None
print(f"After joining divergence: {len(merged):,} rows")

# Compute exact-match pct columns
merged['ens_to_cat_exact_pct'] = np.where(
    merged['n_ensembl_transcripts'] > 0,
    (merged['n_ens_exact'] / merged['n_ensembl_transcripts'] * 100).round(1),
    np.nan
)
merged['cat_to_ens_exact_pct'] = np.where(
    merged['n_cat_transcripts'] > 0,
    (merged['n_cat_exact'] / merged['n_cat_transcripts'] * 100).round(1),
    np.nan
)

s2_full_cols = [
    'assembly_accession', 'ensembl_gene_id', 'cat_gene_id',
    'ensembl_name', 'cat_name', 'ensembl_biotype', 'cat_biotype',
    'frac_ensembl_covered', 'frac_cat_covered',
    'n_ensembl_transcripts', 'n_cat_transcripts',
    'n_ens_exact', 'n_cat_exact',
    'ens_to_cat_exact_pct', 'cat_to_ens_exact_pct',
    'ens_to_cat_concordance_rate', 'cat_to_ens_concordance_rate',
    'avg_jaccard_index',
    'cds_classification', 'start_codon_match', 'stop_codon_match', 'frameshift_detected',
    'divergence_category',
]
s2_full_cols = [c for c in s2_full_cols if c in merged.columns]
out_s2_full = SUPP_DIR / 'supp_table_s2_full_gene_pairs.tsv'
if BUILD_S2_FULL or not out_s2_full.exists():
    print(f"Writing S2 full data deposit: {out_s2_full}")
    merged[s2_full_cols].to_csv(out_s2_full, sep='\t', index=False)
    print(f"Saved S2 full shape: ({len(merged):,}, {len(s2_full_cols)}) ({time.time()-t0:.1f}s)")
else:
    print(f"Reusing existing S2 full data deposit: {out_s2_full}")
    print("Set HPRC_QC_BUILD_S2_FULL=1 to rebuild this 5G file.")

# --- Pre-compute S1 aggregates while rbh_df and merged are in memory ---
# ensembl_name is the HGNC symbol — the only canonical cross-assembly gene key.

n_assemblies_total = rbh_df['assembly_accession'].nunique()

# gp_agg: one row per canonical gene — how many assemblies have it as an RBH pair
gp_agg = (
    rbh_df.groupby('ensembl_name', sort=False)
    .agg(
        n_assemblies_both=('assembly_accession', 'nunique'),
        ensembl_biotype=('ensembl_biotype', safe_mode),
    )
    .reset_index()
    .rename(columns={'ensembl_name': 'gene_name'})
)
gp_agg['n_assemblies_assessed'] = gp_agg['n_assemblies_both']
gp_agg['pct_assemblies_both'] = (gp_agg['n_assemblies_both'] / n_assemblies_total * 100).round(1)
print(f"gp_agg: {len(gp_agg):,} canonical genes ({time.time()-t0:.1f}s)")

# tc_medians: median transcript concordance per canonical gene
tc_medians = (
    merged.dropna(subset=['ensembl_name'])
    .groupby('ensembl_name')
    .agg(
        median_ens_to_cat_exact_pct=('ens_to_cat_exact_pct', 'median'),
        median_cat_to_ens_exact_pct=('cat_to_ens_exact_pct', 'median'),
        median_ens_to_cat_concordance_rate=('ens_to_cat_concordance_rate', 'median'),
        median_cat_to_ens_concordance_rate=('cat_to_ens_concordance_rate', 'median'),
    )
    .reset_index().round(1)
    .rename(columns={'ensembl_name': 'gene_name'})
)
print(f"tc_medians: {len(tc_medians):,} genes ({time.time()-t0:.1f}s)")

# rbh_name_lookup: (assembly_accession, ensembl_gene_id) → gene_name for CI join in cell-10
rbh_name_lookup = (
    rbh_df[['assembly_accession', 'ensembl_gene_id', 'ensembl_name']]
    .drop_duplicates(['assembly_accession', 'ensembl_gene_id'])
    .rename(columns={'ensembl_name': 'gene_name'})
    .reset_index(drop=True)
)
print(f"rbh_name_lookup: {len(rbh_name_lookup):,} entries ({time.time()-t0:.1f}s)")

# Per-assembly base stats — n_pairs and concordance rates, computed while in memory
assm_n_pairs = rbh_df.groupby('assembly_accession').agg(
    n_rbh_gene_pairs=('ensembl_gene_id', 'count'),
    n_protein_coding_rbh=('ensembl_biotype', lambda x: (x == 'protein_coding').sum()),
).reset_index()
assm_tc = (
    merged.groupby('assembly_accession').agg(
        median_ens_to_cat_concordance_rate=('ens_to_cat_concordance_rate', 'median'),
        median_cat_to_ens_concordance_rate=('cat_to_ens_concordance_rate', 'median'),
    ).reset_index().round(3)
)
assm_summary = assm_n_pairs.merge(assm_tc, on='assembly_accession', how='left')
print(f"assm_summary base: {len(assm_summary):,} assemblies ({time.time()-t0:.1f}s)")

del merged, rbh_df, transcript_concordance_df
gc.collect()
print("Large DataFrames freed.")

## Build Supplementary Table S1: Gene-level concordance summary

In [ ]:
t0 = time.time()

# Work with a slim view of divergence_df — only the columns we need
div_slim = divergence_df[[
    'assembly_accession', 'gene_name', 'is_coding',
    'ens_cds_match', 'cat_cds_match',
    'ens_cds_change_type', 'cat_cds_change_type',
    'divergence_category',
]].copy()

# Normalise cds_match booleans (parquet may store as object True/False/None)
div_slim['ens_cds_match'] = div_slim['ens_cds_match'].map({True: 1.0, False: 0.0})
div_slim['cat_cds_match'] = div_slim['cat_cds_match'].map({True: 1.0, False: 0.0})
print(f"div_slim ready: {len(div_slim):,} rows ({time.time()-t0:.1f}s)")

# ── 1. CDS integrity per canonical gene (Ensembl vs CAT agreement on assembly) ──
ci_with_gene = coding_integrity_df.merge(
    rbh_name_lookup, on=['assembly_accession', 'ensembl_gene_id'], how='inner'
)
print(f"ci_with_gene: {len(ci_with_gene):,} rows ({time.time()-t0:.1f}s)")

for col in ['start_codon_match', 'stop_codon_match', 'frameshift_detected']:
    if col in ci_with_gene.columns:
        ci_with_gene[col] = ci_with_gene[col].map(
            {True: True, False: False, 'True': True, 'False': False, 1: True, 0: False}
        )

ci_agg = ci_with_gene.groupby('gene_name').agg(
    n_assemblies_cds_assessed=('assembly_accession', 'nunique'),
    _n_start=('start_codon_match', 'count'), _sum_start=('start_codon_match', 'sum'),
    _n_stop=('stop_codon_match', 'count'),   _sum_stop=('stop_codon_match', 'sum'),
    _n_fs=('frameshift_detected', 'count'),  _sum_fs=('frameshift_detected', 'sum'),
).reset_index()
del ci_with_gene
gc.collect()

for col, n, s in [('pct_start_codon_match', '_n_start', '_sum_start'),
                   ('pct_stop_codon_match',  '_n_stop',  '_sum_stop'),
                   ('pct_frameshift_detected', '_n_fs',  '_sum_fs')]:
    ci_agg[col] = np.where(ci_agg[n] > 0, (ci_agg[s] / ci_agg[n] * 100).round(1), np.nan)
ci_agg = ci_agg[['gene_name', 'n_assemblies_cds_assessed',
                   'pct_start_codon_match', 'pct_stop_codon_match', 'pct_frameshift_detected']]
print(f"ci_agg: {len(ci_agg):,} genes ({time.time()-t0:.1f}s)")

# ── 2. Divergence category pcts per canonical gene (vs GRCh38, Ensembl vs CAT) ──
DIV_CATS = ['both_agree_reference', 'both_agree_diverged',
            'cat_specific_divergence', 'ensembl_specific_divergence', 'insufficient_data']

div_base = div_slim.dropna(subset=['gene_name', 'divergence_category'])
div_counts = (
    div_base.groupby(['gene_name', 'divergence_category'])
    .size().unstack(fill_value=0).reset_index()
)
for cat in DIV_CATS:
    if cat not in div_counts.columns:
        div_counts[cat] = 0
div_counts['_n_div_total'] = div_counts[DIV_CATS].sum(axis=1)
for cat in DIV_CATS:
    div_counts[f'pct_{cat}'] = np.where(
        div_counts['_n_div_total'] > 0,
        (div_counts[cat] / div_counts['_n_div_total'] * 100).round(1), np.nan)
div_mode = (
    div_base.groupby('gene_name')['divergence_category']
    .agg(safe_mode).reset_index()
)
div_mode.columns = ['gene_name', 'predominant_divergence_category']
pct_div_cols = [f'pct_{cat}' for cat in DIV_CATS]
div_final = div_counts[['gene_name'] + pct_div_cols].merge(div_mode, on='gene_name', how='left')
print(f"div_final: {len(div_final):,} genes ({time.time()-t0:.1f}s)")

# ── 3. Reference-anchored CDS quality per gene (each annotator vs GRCh38) ──
# Filter to coding reference genes — non-coding have ens_cds_change_type='non_coding'
div_coding = div_slim[div_slim['is_coding'] == True].dropna(subset=['gene_name']).copy()

# Binary: does each annotator's CDS match GRCh38?
ref_cds_agg = div_coding.groupby('gene_name').agg(
    _n_ens=('ens_cds_match', 'count'), _sum_ens=('ens_cds_match', 'sum'),
    _n_cat=('cat_cds_match', 'count'), _sum_cat=('cat_cds_match', 'sum'),
).reset_index()
ref_cds_agg['pct_ens_cds_match_ref'] = np.where(
    ref_cds_agg['_n_ens'] > 0,
    (ref_cds_agg['_sum_ens'] / ref_cds_agg['_n_ens'] * 100).round(1), np.nan)
ref_cds_agg['pct_cat_cds_match_ref'] = np.where(
    ref_cds_agg['_n_cat'] > 0,
    (ref_cds_agg['_sum_cat'] / ref_cds_agg['_n_cat'] * 100).round(1), np.nan)

# Specific failure modes: coding_lost and frameshift per annotator
div_coding['ens_coding_lost']    = div_coding['ens_cds_change_type'] == 'coding_lost'
div_coding['cat_coding_lost']    = div_coding['cat_cds_change_type'] == 'coding_lost'
div_coding['ens_frameshift_ref'] = div_coding['ens_cds_change_type'].isin(
    ['frameshift_shorter', 'frameshift_longer'])
div_coding['cat_frameshift_ref'] = div_coding['cat_cds_change_type'].isin(
    ['frameshift_shorter', 'frameshift_longer'])

cds_fail_agg = div_coding.groupby('gene_name').agg(
    # Keep annotator-specific denominators. CAT and Ensembl CDS classifications
    # can have different missingness for sparse/immune genes; using the Ensembl
    # denominator for CAT failure sums can create impossible percentages >100.
    _n_ens=('ens_cds_change_type', 'count'),
    _n_cat=('cat_cds_change_type', 'count'),
    _sum_ens_cl=('ens_coding_lost', 'sum'),    _sum_cat_cl=('cat_coding_lost', 'sum'),
    _sum_ens_fs=('ens_frameshift_ref', 'sum'), _sum_cat_fs=('cat_frameshift_ref', 'sum'),
).reset_index()
for col, n, s in [
    ('pct_ens_coding_lost', '_n_ens', '_sum_ens_cl'),
    ('pct_cat_coding_lost', '_n_cat', '_sum_cat_cl'),
    ('pct_ens_frameshift_ref', '_n_ens', '_sum_ens_fs'),
    ('pct_cat_frameshift_ref', '_n_cat', '_sum_cat_fs'),
]:
    cds_fail_agg[col] = np.where(
        cds_fail_agg[n] > 0, (cds_fail_agg[s] / cds_fail_agg[n] * 100).round(1), np.nan)

_bounded_fail_cols = ['pct_ens_coding_lost', 'pct_cat_coding_lost',
                      'pct_ens_frameshift_ref', 'pct_cat_frameshift_ref']
_bad_fail_pct = {
    col: int(((cds_fail_agg[col] < 0) | (cds_fail_agg[col] > 100)).sum())
    for col in _bounded_fail_cols
}
if any(_bad_fail_pct.values()):
    raise ValueError(f"CDS failure percentage outside 0-100 after denominator fix: {_bad_fail_pct}")

# Most common CDS change type per annotator per gene
ens_type_mode = (div_coding.dropna(subset=['ens_cds_change_type'])
    .groupby('gene_name')['ens_cds_change_type'].agg(safe_mode).reset_index()
    .rename(columns={'ens_cds_change_type': 'predominant_ens_cds_change_type'}))
cat_type_mode = (div_coding.dropna(subset=['cat_cds_change_type'])
    .groupby('gene_name')['cat_cds_change_type'].agg(safe_mode).reset_index()
    .rename(columns={'cat_cds_change_type': 'predominant_cat_cds_change_type'}))

# Source rows for the release S3 extensive CDS/reference-discordance table.
# Collapse to one row per gene/assembly before counting recurrence so multi-row
# edge cases cannot inflate assembly-level support. Missing CDS-change calls are
# not treated as non-exact matches.
ens_cds_assessed = div_coding['ens_cds_change_type'].notna()
cat_cds_assessed = div_coding['cat_cds_change_type'].notna()
ens_cds_exact = div_coding['ens_cds_change_type'] == 'exact_match'
cat_cds_exact = div_coding['cat_cds_change_type'] == 'exact_match'
div_coding['has_cds_change_assessment'] = ens_cds_assessed | cat_cds_assessed
div_coding['has_cds_reference_discordance'] = (
    div_coding['has_cds_change_assessment'] & ~(ens_cds_exact & cat_cds_exact)
)
div_coding['ensembl_exact_cat_nonexact'] = ens_cds_exact & cat_cds_assessed & ~cat_cds_exact
div_coding['cat_exact_ensembl_nonexact'] = cat_cds_exact & ens_cds_assessed & ~ens_cds_exact
div_coding['both_nonexact_cds_reference'] = (
    ens_cds_assessed & cat_cds_assessed & ~ens_cds_exact & ~cat_cds_exact
)

cds_reference_gene_assembly_source = (
    div_coding[div_coding['has_cds_change_assessment']]
    .groupby(['gene_name', 'assembly_accession'], dropna=False)
    .agg(
        ensembl_biotype=('ensembl_biotype', safe_mode),
        has_cds_reference_discordance=('has_cds_reference_discordance', 'max'),
        ensembl_exact_cat_nonexact=('ensembl_exact_cat_nonexact', 'max'),
        cat_exact_ensembl_nonexact=('cat_exact_ensembl_nonexact', 'max'),
        both_nonexact_cds_reference=('both_nonexact_cds_reference', 'max'),
        cat_coding_lost=('cat_coding_lost', 'max'),
        ensembl_coding_lost=('ens_coding_lost', 'max'),
        n_raw_gene_pair_observations=('assembly_accession', 'size'),
    )
    .reset_index()
)

cds_change_pair_counts = (
    div_coding
    .dropna(subset=['gene_name', 'ens_cds_change_type', 'cat_cds_change_type'])
    .groupby(['gene_name', 'ens_cds_change_type', 'cat_cds_change_type'])
    .size()
    .reset_index(name='n_raw_observations_for_predominant_cds_change_pair')
    .sort_values(['gene_name', 'n_raw_observations_for_predominant_cds_change_pair',
                  'ens_cds_change_type', 'cat_cds_change_type'],
                 ascending=[True, False, True, True])
    .drop_duplicates('gene_name')
    .rename(columns={
        'ens_cds_change_type': 'predominant_pair_ensembl_cds_change_type',
        'cat_cds_change_type': 'predominant_pair_cat_cds_change_type',
    })
)
print(f"S3 CDS/reference source: {len(cds_reference_gene_assembly_source):,} gene-assembly rows; "
      f"{cds_reference_gene_assembly_source['gene_name'].nunique():,} genes ({time.time()-t0:.1f}s)")
del div_coding
gc.collect()

ref_cds_agg = (
    ref_cds_agg[['gene_name', 'pct_ens_cds_match_ref', 'pct_cat_cds_match_ref']]
    .merge(cds_fail_agg[['gene_name', 'pct_ens_coding_lost', 'pct_cat_coding_lost',
                          'pct_ens_frameshift_ref', 'pct_cat_frameshift_ref']], on='gene_name', how='left')
    .merge(ens_type_mode, on='gene_name', how='left')
    .merge(cat_type_mode, on='gene_name', how='left')
)
del cds_fail_agg, ens_type_mode, cat_type_mode
gc.collect()
print(f"ref_cds_agg: {len(ref_cds_agg):,} coding genes ({time.time()-t0:.1f}s)")

# ── 4. S2: Per-assembly summary (462 rows — the actual supplement) ──
# CDS full-match rate per assembly (exclude No_CDS rows from denominator)
ci_cls = coding_integrity_df.groupby(['assembly_accession', 'classification']).size().unstack(fill_value=0).reset_index()
ci_cls.columns.name = None
ci_class_cols = [c for c in ci_cls.columns if c != 'assembly_accession']
ci_with_cds = [c for c in ci_class_cols if c != 'No_CDS']
ci_cls['_n_with_cds'] = ci_cls[[c for c in ci_with_cds]].sum(axis=1)
ci_cls['pct_full_cds_match'] = np.where(
    ci_cls['_n_with_cds'] > 0,
    (ci_cls.get('Full_Match', 0) / ci_cls['_n_with_cds'] * 100).round(1), np.nan)
assm_ci = ci_cls[['assembly_accession', 'pct_full_cds_match']]
del ci_cls
gc.collect()

# Divergence category pcts per assembly
assm_div = (
    div_slim.dropna(subset=['divergence_category'])
    .groupby(['assembly_accession', 'divergence_category'])
    .size().unstack(fill_value=0).reset_index()
)
assm_div.columns.name = None
for cat in DIV_CATS:
    if cat not in assm_div.columns:
        assm_div[cat] = 0
assm_div['_n_div_total'] = assm_div[DIV_CATS].sum(axis=1)
for cat in DIV_CATS:
    assm_div[f'pct_{cat}'] = np.where(
        assm_div['_n_div_total'] > 0,
        (assm_div[cat] / assm_div['_n_div_total'] * 100).round(1), np.nan)
assm_div = assm_div[['assembly_accession'] + [f'pct_{cat}' for cat in DIV_CATS]]

del div_slim
gc.collect()

assm_summary = (
    assm_summary
    .merge(assm_ci,  on='assembly_accession', how='left')
    .merge(assm_div, on='assembly_accession', how='left')
    .sort_values('assembly_accession').reset_index(drop=True)
)
del assm_ci, assm_div
gc.collect()

out_s2 = SUPP_DIR / 'supp_table_s2_per_assembly_summary.tsv'
assm_summary.to_csv(out_s2, sep='\t', index=False)
print(f"Saved S2: {out_s2}")
print(f"S2 shape: {assm_summary.shape} ({time.time()-t0:.1f}s)")

# ── 5. Assemble S1 (enriched) ──
s1_df = gp_agg.copy()
s1_df = s1_df.merge(tc_medians,  on='gene_name', how='left')
s1_df = s1_df.merge(ci_agg,      on='gene_name', how='left')
s1_df = s1_df.merge(div_final,   on='gene_name', how='left')
s1_df = s1_df.merge(ref_cds_agg, on='gene_name', how='left')
s1_df['predominant_divergence_category'] = s1_df['predominant_divergence_category'].fillna('N/A')

s1_col_order = [
    'gene_name', 'ensembl_biotype',
    'n_assemblies_assessed', 'n_assemblies_both', 'pct_assemblies_both',
    # Transcript concordance: Ensembl vs CAT agreement on the assembly
    'median_ens_to_cat_exact_pct', 'median_cat_to_ens_exact_pct',
    'median_ens_to_cat_concordance_rate', 'median_cat_to_ens_concordance_rate',
    # CDS integrity: Ensembl vs CAT agreement on the assembly (do they call the same CDS?)
    'n_assemblies_cds_assessed',
    'pct_start_codon_match', 'pct_stop_codon_match', 'pct_frameshift_detected',
    # Reference-anchored CDS quality: each annotator independently vs GRCh38
    'pct_ens_cds_match_ref', 'pct_cat_cds_match_ref',
    'pct_ens_coding_lost', 'pct_cat_coding_lost',
    'pct_ens_frameshift_ref', 'pct_cat_frameshift_ref',
    'predominant_ens_cds_change_type', 'predominant_cat_cds_change_type',
    # GRCh38 divergence: do both annotators agree, and in which direction?
    'predominant_divergence_category',
    'pct_both_agree_reference', 'pct_both_agree_diverged',
    'pct_cat_specific_divergence', 'pct_ensembl_specific_divergence',
    'pct_insufficient_data',
]
s1_col_order = [c for c in s1_col_order if c in s1_df.columns]
s1_df = s1_df[s1_col_order].sort_values('gene_name').reset_index(drop=True)

out_s1 = SUPP_DIR / 'supp_table_s1_gene_concordance_summary.tsv'
s1_df.to_csv(out_s1, sep='\t', index=False)
print(f"\nSaved S1: {out_s1}")
print(f"S1 shape: {s1_df.shape} ({time.time()-t0:.1f}s total)")
s1_df.head(3)


## Summary

## Build Supplementary Table S3: extensive CDS/reference-discordant figure-inset genes

S3 is the extensive gene-level source table for the CDS/reference-discordance inset. It is **not** the recurrent thresholded subset; the threshold ladder is emitted separately as supporting QC.


In [ ]:
t0 = time.time()

# S3: extensive protein-coding genes represented in the CDS/reference-discordant
# figure inset/source set. Inclusion is at least one assessed assembly where CAT
# and Ensembl are not both exact CDS matches to GRCh38.
s3_gene_assembly = cds_reference_gene_assembly_source.copy()
for col in ['has_cds_reference_discordance', 'ensembl_exact_cat_nonexact',
            'cat_exact_ensembl_nonexact', 'both_nonexact_cds_reference',
            'cat_coding_lost', 'ensembl_coding_lost']:
    s3_gene_assembly[col] = s3_gene_assembly[col].fillna(False).astype(bool)

s3_summary = (
    s3_gene_assembly
    .groupby('gene_name')
    .agg(
        ensembl_biotype=('ensembl_biotype', safe_mode),
        n_assemblies_assessed=('assembly_accession', 'nunique'),
        n_cds_reference_discordant_assemblies=('has_cds_reference_discordance', 'sum'),
        n_ensembl_exact_cat_nonexact_assemblies=('ensembl_exact_cat_nonexact', 'sum'),
        n_cat_exact_ensembl_nonexact_assemblies=('cat_exact_ensembl_nonexact', 'sum'),
        n_both_nonexact_cds_reference_assemblies=('both_nonexact_cds_reference', 'sum'),
        n_cat_coding_lost_assemblies=('cat_coding_lost', 'sum'),
        n_ensembl_coding_lost_assemblies=('ensembl_coding_lost', 'sum'),
    )
    .reset_index()
)
s3_summary = s3_summary[s3_summary['n_cds_reference_discordant_assemblies'] > 0].copy()

for count_col, pct_col in [
    ('n_cds_reference_discordant_assemblies', 'pct_cds_reference_discordant_assemblies'),
    ('n_ensembl_exact_cat_nonexact_assemblies', 'pct_ensembl_exact_cat_nonexact_assemblies'),
    ('n_cat_exact_ensembl_nonexact_assemblies', 'pct_cat_exact_ensembl_nonexact_assemblies'),
    ('n_both_nonexact_cds_reference_assemblies', 'pct_both_nonexact_cds_reference_assemblies'),
    ('n_cat_coding_lost_assemblies', 'pct_cat_coding_lost_assemblies'),
    ('n_ensembl_coding_lost_assemblies', 'pct_ensembl_coding_lost_assemblies'),
]:
    s3_summary[pct_col] = np.where(
        s3_summary['n_assemblies_assessed'] > 0,
        (s3_summary[count_col] / s3_summary['n_assemblies_assessed'] * 100).round(4),
        np.nan,
    )

# Optional GRCh38 locus context. This cache is available in local release builds;
# if absent on HPC, S3 remains valid but release QC will warn about missing locus columns.
gene_positions_cache = CACHE_DIR / 'gene_positions_grch38.parquet'
if gene_positions_cache.exists():
    gene_positions = pd.read_parquet(gene_positions_cache)
    pos_rename = {
        'chrom': 'grch38_chrom',
        'start': 'grch38_start',
        'end': 'grch38_end',
        'midpoint': 'grch38_midpoint',
        'strand': 'grch38_strand',
    }
    gene_positions = gene_positions.rename(columns=pos_rename)
    pos_cols = ['gene_name'] + [c for c in pos_rename.values() if c in gene_positions.columns]
    s3_summary = s3_summary.merge(gene_positions[pos_cols].drop_duplicates('gene_name'),
                                  on='gene_name', how='left')
    print(f"Merged GRCh38 locus context from {gene_positions_cache}")
else:
    print(f"WARNING: {gene_positions_cache} not found; S3 will omit GRCh38 locus columns")

s3_summary = (
    s3_summary
    .merge(cds_change_pair_counts, on='gene_name', how='left')
    .merge(s1_df[['gene_name', 'pct_ens_cds_match_ref', 'pct_cat_cds_match_ref',
                  'pct_ens_coding_lost', 'pct_cat_coding_lost',
                  'predominant_ens_cds_change_type', 'predominant_cat_cds_change_type',
                  'predominant_divergence_category']],
           on='gene_name', how='left')
)

s3_cols = [
    'gene_name', 'ensembl_biotype',
    'grch38_chrom', 'grch38_start', 'grch38_end', 'grch38_midpoint', 'grch38_strand',
    'n_assemblies_assessed', 'n_cds_reference_discordant_assemblies',
    'pct_cds_reference_discordant_assemblies',
    'n_ensembl_exact_cat_nonexact_assemblies', 'n_cat_exact_ensembl_nonexact_assemblies',
    'n_both_nonexact_cds_reference_assemblies',
    'n_cat_coding_lost_assemblies', 'n_ensembl_coding_lost_assemblies',
    'pct_ensembl_exact_cat_nonexact_assemblies', 'pct_cat_exact_ensembl_nonexact_assemblies',
    'pct_both_nonexact_cds_reference_assemblies',
    'pct_cat_coding_lost_assemblies', 'pct_ensembl_coding_lost_assemblies',
    'predominant_ensembl_cds_change_type', 'predominant_cat_cds_change_type',
    'predominant_pair_ensembl_cds_change_type', 'predominant_pair_cat_cds_change_type',
    'n_raw_observations_for_predominant_cds_change_pair',
    'pct_ensembl_cds_match_reference', 'pct_cat_cds_match_reference',
    'pct_ensembl_coding_lost', 'pct_cat_coding_lost',
    'predominant_divergence_category',
]
s3_summary = s3_summary.rename(columns={
    'pct_ens_cds_match_ref': 'pct_ensembl_cds_match_reference',
    'pct_cat_cds_match_ref': 'pct_cat_cds_match_reference',
    'pct_ens_coding_lost': 'pct_ensembl_coding_lost',
    'predominant_ens_cds_change_type': 'predominant_ensembl_cds_change_type',
})
s3_cols = [c for c in s3_cols if c in s3_summary.columns]
s3_df = (
    s3_summary[s3_cols]
    .sort_values(['pct_cds_reference_discordant_assemblies', 'n_assemblies_assessed', 'gene_name'],
                 ascending=[False, False, True])
    .reset_index(drop=True)
)

out_s3 = SUPP_DIR / 'supp_table_s3_cds_reference_discordant_genes.tsv'
s3_df.to_csv(out_s3, sep='\t', index=False)
print(f"Saved release S3: {out_s3}")
print(f"S3 shape: {s3_df.shape} ({time.time()-t0:.1f}s)")

# Supporting recurrence threshold ladder, not the primary S3 table.
threshold_rows = []
for threshold in [0.0000001, 1, 5, 10, 20, 50, 100]:
    eligible = s3_df['pct_cds_reference_discordant_assemblies'] >= threshold
    eligible_ge50 = eligible & (s3_df['n_assemblies_assessed'] >= 50)
    threshold_rows.append({
        'threshold_pct_cds_reference_discordant_assemblies': threshold,
        'n_genes': int(eligible.sum()),
        'n_genes_with_n_assemblies_assessed_ge_50': int(eligible_ge50.sum()),
        'median_n_assemblies_assessed': float(s3_df.loc[eligible, 'n_assemblies_assessed'].median()) if eligible.any() else np.nan,
        'median_n_cds_reference_discordant_assemblies': float(s3_df.loc[eligible, 'n_cds_reference_discordant_assemblies'].median()) if eligible.any() else np.nan,
    })
s3_threshold_df = pd.DataFrame(threshold_rows)
out_s3_thresholds = SUPP_DIR / 'supp_table_s3_recurrence_threshold_sensitivity.tsv'
s3_threshold_df.to_csv(out_s3_thresholds, sep='\t', index=False)
print(f"Saved S3 threshold sensitivity: {out_s3_thresholds}")
print(s3_threshold_df.to_string(index=False))

# Keep the old thresholded discordance list as a diagnostic, not as release S3.
DISCORD_THRESHOLD = 10.0
CDS_DIFF_THRESHOLD = 20.0
MIN_ASSEMBLIES = 50
s3_base = s1_df[s1_df['n_assemblies_assessed'] >= MIN_ASSEMBLIES].copy()
mask_div = (
    (s3_base['pct_cat_specific_divergence'].fillna(0) >= DISCORD_THRESHOLD) |
    (s3_base['pct_ensembl_specific_divergence'].fillna(0) >= DISCORD_THRESHOLD)
)
mask_cds_lost = s3_base['pct_cat_coding_lost'].fillna(0) >= DISCORD_THRESHOLD
mask_cds_ref = (
    s3_base['pct_ens_cds_match_ref'].notna() &
    s3_base['pct_cat_cds_match_ref'].notna() &
    ((s3_base['pct_ens_cds_match_ref'] - s3_base['pct_cat_cds_match_ref']).abs() >= CDS_DIFF_THRESHOLD)
)
s3_thresholded_df = s3_base[mask_div | mask_cds_lost | mask_cds_ref].copy()
def discordance_type(row):
    types = []
    if pd.notna(row.get('pct_cat_specific_divergence')) and row['pct_cat_specific_divergence'] >= DISCORD_THRESHOLD:
        types.append('cat_diverges')
    if pd.notna(row.get('pct_ensembl_specific_divergence')) and row['pct_ensembl_specific_divergence'] >= DISCORD_THRESHOLD:
        types.append('ensembl_diverges')
    if pd.notna(row.get('pct_cat_coding_lost')) and row['pct_cat_coding_lost'] >= DISCORD_THRESHOLD:
        types.append('cat_coding_lost')
    ens_ref = row.get('pct_ens_cds_match_ref')
    cat_ref = row.get('pct_cat_cds_match_ref')
    if pd.notna(ens_ref) and pd.notna(cat_ref) and abs(ens_ref - cat_ref) >= CDS_DIFF_THRESHOLD:
        types.append('cds_ref_discordant')
    return '|'.join(types)
s3_thresholded_df.insert(2, 'discordance_type', s3_thresholded_df.apply(discordance_type, axis=1))
s3_thresholded_df = s3_thresholded_df.sort_values(
    ['pct_cat_specific_divergence', 'pct_cat_coding_lost'], ascending=False
).reset_index(drop=True)
out_s3_thresholded = SUPP_DIR / 'supp_table_s3_thresholded_discordant_genes_diagnostic.tsv'
s3_thresholded_df.to_csv(out_s3_thresholded, sep='\t', index=False)
print(f"Saved diagnostic thresholded list: {out_s3_thresholded} ({len(s3_thresholded_df):,} rows)")
s3_df.head(10)


In [ ]:
print(f"Table S1: {len(s1_df):,} genes × {len(s1_df.columns)} columns  (per-gene concordance summary)")
print(f"Table S2: {len(assm_summary):,} assemblies × {len(assm_summary.columns)} columns  (per-assembly summary)")
print(f"Table S3: {len(s3_df):,} genes × {len(s3_df.columns)} columns  (extensive CDS/reference-discordant inset source)")
print(f"Diagnostic thresholded list: {len(s3_thresholded_df):,} genes × {len(s3_thresholded_df.columns)} columns")
print(f"\nS3 threshold sensitivity:")
print(s3_threshold_df.to_string(index=False))
print(f"\nFiles written to: {SUPP_DIR}")
for f in sorted(SUPP_DIR.glob('*.tsv')):
    size = f.stat().st_size / 1024**2
    print(f"  {f.name}: {size:.1f} MB")


## Pre-release QC checks and plots

Run this section before packaging release materials. It writes machine-readable QC summaries and simple PNG diagnostics under `supplementary_tables/pre_release_qc/`.


In [ ]:
import matplotlib.pyplot as plt

qc_out = SUPP_DIR / 'pre_release_qc'
qc_out.mkdir(parents=True, exist_ok=True)

release_tables = {
    'S1': s1_df,
    'S2': assm_summary,
    'S3': s3_df,
    'S3_thresholded_diagnostic': s3_thresholded_df,
}

bounded_rows = []
for table_name, df in release_tables.items():
    for col in df.columns:
        lc = col.lower()
        if (col.startswith('pct_') or ('_pct' in col) or (col.startswith('median_') and ('pct' in col or 'rate' in col))) and 'delta' not in lc:
            vals = pd.to_numeric(df[col], errors='coerce')
            if vals.notna().any():
                bad_mask = (vals < 0) | (vals > 100)
                bounded_rows.append({
                    'table': table_name,
                    'column': col,
                    'n_non_null': int(vals.notna().sum()),
                    'min': float(vals.min()),
                    'max': float(vals.max()),
                    'n_outside_0_100': int(bad_mask.sum()),
                })
bounded_qc = pd.DataFrame(bounded_rows)
bounded_qc.to_csv(qc_out / 'bounded_percentage_qc.tsv', sep='\t', index=False)

missing_rows = []
for table_name, df in release_tables.items():
    for col in df.columns:
        s = df[col]
        missing_rows.append({
            'table': table_name,
            'column': col,
            'n_rows': len(df),
            'n_blank_or_na': int(s.isna().sum() + s.astype(str).str.strip().isin(['', 'NA', 'N/A']).sum()),
        })
missing_qc = pd.DataFrame(missing_rows)
missing_qc.to_csv(qc_out / 'missingness_qc.tsv', sep='\t', index=False)

# S2 release-specific checks: all expected assemblies present, no missing divergence percentages.
expected_n_assemblies = 462
known_symbol_rerun_assemblies = ['GCA_046332455.1', 'GCA_046332475.1', 'GCA_046332485.1', 'GCA_046332495.1']
s2_div_cols = ['pct_both_agree_reference', 'pct_both_agree_diverged',
               'pct_cat_specific_divergence', 'pct_ensembl_specific_divergence',
               'pct_insufficient_data']
s2_checks = []
s2_checks.append({'check': 'n_assemblies', 'value': assm_summary['assembly_accession'].nunique(),
                  'expected': expected_n_assemblies, 'pass': assm_summary['assembly_accession'].nunique() == expected_n_assemblies})
s2_missing_div = assm_summary[s2_div_cols].isna().any(axis=1)
s2_checks.append({'check': 'assemblies_with_missing_divergence_pct', 'value': int(s2_missing_div.sum()),
                  'expected': 0, 'pass': int(s2_missing_div.sum()) == 0})
for asm in known_symbol_rerun_assemblies:
    row = assm_summary[assm_summary['assembly_accession'] == asm]
    present = len(row) == 1
    div_complete = bool(present and row[s2_div_cols].notna().all(axis=1).iloc[0])
    s2_checks.append({'check': f'{asm}_present_with_divergence_pct', 'value': div_complete,
                      'expected': True, 'pass': div_complete})
s2_check_df = pd.DataFrame(s2_checks)
s2_check_df.to_csv(qc_out / 's2_release_checks.tsv', sep='\t', index=False)

# S3 release-specific checks: extensive table should match source gene count and should not be the thresholded diagnostic list.
s3_checks = pd.DataFrame([
    {'check': 's3_rows', 'value': len(s3_df), 'expected': '>= thresholded diagnostic rows', 'pass': len(s3_df) >= len(s3_thresholded_df)},
    {'check': 's3_has_required_discordance_count', 'value': 'n_cds_reference_discordant_assemblies' in s3_df.columns, 'expected': True, 'pass': 'n_cds_reference_discordant_assemblies' in s3_df.columns},
    {'check': 's3_min_discordant_assemblies', 'value': int(s3_df['n_cds_reference_discordant_assemblies'].min()), 'expected': '>=1', 'pass': int(s3_df['n_cds_reference_discordant_assemblies'].min()) >= 1},
    {'check': 's3_has_grch38_locus_columns', 'value': all(c in s3_df.columns for c in ['grch38_chrom', 'grch38_start', 'grch38_end']), 'expected': True, 'pass': all(c in s3_df.columns for c in ['grch38_chrom', 'grch38_start', 'grch38_end'])},
])
s3_checks.to_csv(qc_out / 's3_release_checks.tsv', sep='\t', index=False)

# S2 full data-deposit missingness: this table intentionally contains ID-only rows;
# quantify them so release notes can document the denominator.
s2_full_path = SUPP_DIR / 'supp_table_s2_full_gene_pairs.tsv'
full_total = full_ens_blank = full_cat_blank = full_div_blank = 0
for chunk in pd.read_csv(s2_full_path, sep='\t', dtype=str, keep_default_na=False,
                         usecols=['ensembl_name', 'cat_name', 'divergence_category'], chunksize=500_000):
    full_total += len(chunk)
    full_ens_blank += int(chunk['ensembl_name'].astype(str).str.strip().eq('').sum())
    full_cat_blank += int(chunk['cat_name'].astype(str).str.strip().eq('').sum())
    full_div_blank += int(chunk['divergence_category'].astype(str).str.strip().eq('').sum())
s2_full_missingness = pd.DataFrame([
    {'metric': 'n_rows', 'value': full_total, 'pct_of_rows': 100.0},
    {'metric': 'blank_ensembl_name', 'value': full_ens_blank, 'pct_of_rows': round(full_ens_blank / full_total * 100, 3)},
    {'metric': 'blank_cat_name', 'value': full_cat_blank, 'pct_of_rows': round(full_cat_blank / full_total * 100, 3)},
    {'metric': 'blank_divergence_category', 'value': full_div_blank, 'pct_of_rows': round(full_div_blank / full_total * 100, 3)},
])
s2_full_missingness.to_csv(qc_out / 's2_full_gene_pairs_missingness.tsv', sep='\t', index=False)

# Aggregate pass/fail summary.
qc_summary = pd.concat([
    pd.DataFrame([{'check': 'bounded_percentages_outside_0_100',
                   'value': int(bounded_qc['n_outside_0_100'].sum()),
                   'expected': 0,
                   'pass': int(bounded_qc['n_outside_0_100'].sum()) == 0}]),
    s2_check_df[['check', 'value', 'expected', 'pass']],
    s3_checks[['check', 'value', 'expected', 'pass']],
], ignore_index=True)
qc_summary.to_csv(qc_out / 'pre_release_qc_summary.tsv', sep='\t', index=False)
print(qc_summary.to_string(index=False))
print(f"QC files written to {qc_out}")
if not qc_summary['pass'].all():
    print("WARNING: one or more pre-release QC checks failed; inspect TSVs above before packaging.")

# Plain diagnostic plots.
plt.figure(figsize=(7, 4))
assm_summary[s2_div_cols].apply(pd.to_numeric).boxplot(rot=35)
plt.ylabel('Percent of gene-pair observations')
plt.title('S2 divergence percentages across assemblies')
plt.tight_layout()
plt.savefig(qc_out / 'plot_s2_divergence_percentage_boxplot.png', dpi=200)
plt.show()

plt.figure(figsize=(7, 4))
s3_df['predominant_divergence_category'].value_counts().plot(kind='bar', color='black')
plt.ylabel('Genes')
plt.title('S3 extensive table: predominant divergence category')
plt.tight_layout()
plt.savefig(qc_out / 'plot_s3_predominant_divergence_category.png', dpi=200)
plt.show()

plt.figure(figsize=(7, 4))
plot_rows = bounded_qc.groupby('table')['n_outside_0_100'].sum().reindex(release_tables.keys()).fillna(0)
plot_rows.plot(kind='bar', color='black')
plt.ylabel('Cells outside 0-100')
plt.title('Bounded percentage QC by release table')
plt.tight_layout()
plt.savefig(qc_out / 'plot_bounded_percentage_failures.png', dpi=200)
plt.show()

